# Chapter 6.10: From NumPy to Keras

Back to the course site: https://purwar-lab.github.io/ml-for-robotics-

This notebook translates the pure NumPy sin(x) network into TensorFlow/Keras. The architecture, Adam optimizer, and MSE loss are the same; Keras handles the training loop automatically.

## Cell 1 - Generate sin(x) data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

np.random.seed(42)
x = np.linspace(0, 2 * np.pi, 1000).reshape(-1, 1)
y = np.sin(x)

X_train, X_val, y_train, y_val = train_test_split(
    x, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}  Val: {X_val.shape}")


## Cell 2 - Build the Keras sine network

In [ ]:
import tensorflow as tf
from tensorflow import keras

LEARNING_RATE = 0.001
EPOCHS = 3000
HIDDEN_DIM1 = 20
HIDDEN_DIM2 = 10

model = keras.Sequential([
    keras.layers.Input(shape=(1,)),
    keras.layers.Dense(HIDDEN_DIM1, activation="relu"),
    keras.layers.Dense(HIDDEN_DIM2, activation="relu"),
    keras.layers.Dense(1),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mse",
    metrics=["mae"],
)

model.summary()


## Cell 3 - Train and plot

Run this once, then change the configuration in Cell 2 and rerun Cells 2 and 3.

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=32,
    verbose=0,
)

x_plot = np.linspace(0, 4 * np.pi, 1000).reshape(-1, 1)
y_true_plot = np.sin(x_plot)
y_pred_plot = model.predict(x_plot, verbose=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history.history["loss"], label="Train loss")
ax1.plot(history.history["val_loss"], label="Val loss", linestyle="--")
ax1.set_title("MSE Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(x_plot, y_true_plot, label="Actual sin(x)")
ax2.plot(x_plot, y_pred_plot, label="Keras prediction", linestyle="--")
ax2.axvline(2 * np.pi, color="gray", linestyle=":", label="End of training range")
ax2.set_title("Generalization: 0 to 4pi")
ax2.set_xlabel("x")
ax2.set_ylabel("sin(x)")
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Final val loss: {history.history['val_loss'][-1]:.6f}")
print(f"Final val MAE:  {history.history['val_mae'][-1]:.6f}")


## Cell 4 - Fix it progressively

Rerun Cells 2 and 3 after each change.

1. Try fewer neurons: `HIDDEN_DIM1 = 5`, `HIDDEN_DIM2 = 2`.
2. Try more neurons: `HIDDEN_DIM1 = 100`, `HIDDEN_DIM2 = 50`.
3. Try fewer epochs: `EPOCHS = 300`.
4. Try learning rates `0.0001`, `0.001`, and `0.01`.

## Cell 5 - Add dropout and early stopping

In [ ]:
LEARNING_RATE = 0.001
EPOCHS = 5000
HIDDEN_DIM1 = 64
HIDDEN_DIM2 = 32
DROPOUT_RATE = 0.1

model2 = keras.Sequential([
    keras.layers.Input(shape=(1,)),
    keras.layers.Dense(HIDDEN_DIM1, activation="relu"),
    keras.layers.Dropout(DROPOUT_RATE),
    keras.layers.Dense(HIDDEN_DIM2, activation="relu"),
    keras.layers.Dropout(DROPOUT_RATE),
    keras.layers.Dense(1),
])

model2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mse",
    metrics=["mae"],
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=150,
    restore_best_weights=True,
)

history2 = model2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)

stopped_at = len(history2.history["loss"])
print(f"Stopped at epoch {stopped_at} / {EPOCHS}")
print(f"Final val loss: {history2.history['val_loss'][-1]:.6f}")
print(f"Final val MAE:  {history2.history['val_mae'][-1]:.6f}")

x_plot = np.linspace(0, 4 * np.pi, 1000).reshape(-1, 1)
y_true_plot = np.sin(x_plot)
y_pred_plot = model2.predict(x_plot, verbose=0)

plt.figure(figsize=(10, 5))
plt.plot(x_plot, y_true_plot, label="Actual sin(x)")
plt.plot(x_plot, y_pred_plot, label="Dropout + early stopping prediction", linestyle="--")
plt.axvline(2 * np.pi, color="gray", linestyle=":", label="End of training range")
plt.title("Regularized Keras Model")
plt.xlabel("x")
plt.ylabel("sin(x)")
plt.legend()
plt.show()
